# QAChat dogfooding: does it answer questions about its own code correctly?

There's no published benchmark table for this one, unlike the other notebooks in this series -- see
the chat discussion this notebook grew out of. That's not a gap to apologize for: this notebook asks a
different kind of question than 10-60 do, and it has to earn its own credibility rather than borrow
one from a leaderboard.

**The Problem.** Retrieval quality (10-50) and generation-with-injected-context (60) both stop short
of testing the thing QAChat's system prompt actually promises: answer only from the supplied evidence,
cite claims with `[1]`, `[2]`, and say when the evidence is insufficient. Does QAChat's real,
end-to-end chat pipeline -- retrieval, prompt construction, generation, all of it -- actually honor
that contract when asked real questions about a real codebase?

**The Data.** QAChat's own `qachat/` package *and* its `tutorials/` notebooks, opened as a real
`kind=code` project against the live repository via `prepare_project_workspace` -- not a synthetic or
hand-copied corpus like the earlier notebooks had to build. 20 hand-authored questions
(`benchmarks/gold/dogfood_qachat_code.json`), each with a citation verified against the actual source
and a fact checklist for grading, plus a reference reply written from that checklist as a comparison
aid for Tier 2 grading. Six are deliberate negatives: questions the codebase doesn't actually answer,
where the correct response is "insufficient evidence," not a confident guess -- two phrased as a
missing fact ("what's the largest corpus size...") and four phrased as a yes/no judgment ("does
QAChat encrypt..."), added after the first run's two fact-style negatives both failed, to check
whether that failure was specific to the question's shape or a more general pattern. Four more
(A17-A20) were added later, targeting modules the first 16 barely touched (deterministic
retrieval-context expansion, the GUI model catalog, and backend selection). Two more (A21-A22) target
`tutorials/` content specifically -- documented usage and format conventions that live in the
tutorial notebooks' prose, not necessarily stated as clearly (or at all) in `qachat/`'s own docstrings
-- since that's the main thing indexing `tutorials/` can test that the package alone couldn't.

**The Metrics.** Two tiers, not one score. Tier 1 is fully mechanical and runs automatically in this
notebook: citation validity (every `[n]` in an answer refers to evidence QAChat actually supplied),
citation coverage (of a question's gold-relevant evidence, how much actually got cited), abstention
correctness on the negative questions, and verbatim-copy detection (the system prompt forbids
copying source text directly). Tier 2 -- does each cited claim actually hold up, does the answer
satisfy its fact checklist -- needs judgment a regex can't provide, and this notebook exports it as a
worksheet for you to grade by hand rather than wiring in an LLM-judge API call.

**What's different from the other notebooks.** This one calls `RAGSession.ask()` directly -- QAChat's
real top-level API, the same one the GUI uses -- rather than hand-rebuilding retrieval and prompt
construction like notebook 60 does. `RAGSession.__init__` alone handles opening the project, loading
both models, and building or reusing the index, so there's no separate "load models" / "build index"
section here the way 10-60 need one. `RAGSession.reset()` runs before every question, so each one is
an independent single-turn conversation -- otherwise QAChat's normal multi-turn memory would let later
answers lean on earlier questions, which isn't what's being measured here. And because the corpus is
QAChat's own already-existing source tree, there's no HuggingFace dataset to load and no corpus to
write out by hand -- Section 3 stages `qachat/` and `tutorials/` together and `prepare_project_workspace`
mirrors that into a managed workspace itself, tracking changes the same way the actual product does
when reopened. Section 5 overrides the chat session's system prompt before any question is asked: an
earlier run of this notebook (with a no-evidence control, since removed) showed the abstention
failures were a prompt-following gap fixable by a more explicit instruction, not a raw capability gap
needing a bigger chat model -- so that fix is applied directly to the real run below instead.

**A note on the baseline.** Every prior run of this notebook (including the last clean 18/20 result)
scored the package-only corpus. Section 3 now indexes `tutorials/` too, on a fresh workspace, so the
next run's `chunk_count` and scores aren't directly comparable to that history -- this is a genuinely
different, larger corpus, not a rerun of the same one.


## 1. Prepare QAChat

Locate the QAChat repo root, put it on the import path, and run QAChat's notebook setup.


In [1]:
from pathlib import Path

root = Path.cwd()
while not (root / "qachat").is_dir() and root != root.parent:
    root = root.parent

print(f"QAChat root: {root}")


QAChat root: /Users/cmanzo/Documents/GitHub/QAChat


In [2]:
import sys

if str(root) not in sys.path:
    sys.path.insert(0, str(root))


In [3]:
from setup.notebook import prepare_notebook

root = prepare_notebook(root)


QAChat environment ready: /Users/cmanzo/Documents/GitHub/QAChat/.venv/bin/python


## 2. Load the gold question set

No HuggingFace dataset here -- the questions and their gold citations are hand-authored and checked
into the repo (`benchmarks/gold/dogfood_qachat_code.json`), since only a human who actually knows this
codebase can write trustworthy ground truth for it. `load_generation_tasks` (`benchmarks/common.py`)
turns each entry into a `GenerationTask`.


In [4]:
from benchmarks.common import load_generation_tasks

gold_path = root / "benchmarks" / "gold" / "dogfood_qachat_code.json"
tasks = load_generation_tasks(gold_path)

abstention_count = sum(1 for task in tasks if task.requires_abstention)
print(f"Loaded {len(tasks)} questions ({abstention_count} deliberate abstention questions).")
for task in tasks:
    marker = " [abstention]" if task.requires_abstention else ""
    print(f"  {task.task_id}{marker}: {task.question}")


Loaded 16 questions (6 deliberate abstention questions).
  A1: Why does chunk_source_units skip a source unit whose text is empty after stripping?
  A2: What does code_search actually match retrieval candidates on?
  A3: What is QAChat's RAG system prompt, word for word?
  A4: How does RAGSession build the evidence block it hands to the model -- what does each numbered citation actually look like?
  A5: Does hybrid_search use Reciprocal Rank Fusion, and what's the rank constant?
  A6: What happens if a project folder has more files than scan_repository's max_entries?
  A7: Does EmbeddingSession itself have a batch_size parameter, or does batching live somewhere else?
  A8: What are the default retrieval count and generation max_new_tokens in QAChat's shipped profile template?
  A9: What actually changes between a kind=literature project and a kind=code project?
  A10: What exactly does indexes_are_current fingerprint to decide a saved index is stale?
  A11 [abstention]: What's the larg

/Users/cmanzo/Documents/GitHub/QAChat/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. Point QAChat at its own repository -- package and tutorials

`prepare_project_workspace` is the real function QAChat uses to open an existing folder as a project --
not a benchmark-only shortcut. It takes one `source_directory`, so covering both `qachat/` (the package)
and `tutorials/` (QAChat's own 13 usage-tutorial notebooks -- real, intended-usage knowledge that isn't
all in the package's own docstrings, the same reasoning notebook 90 applies to DeepTrack2) needs a small
staging step first: the two folders are siblings at the repo root, alongside `benchmarks/`, `tests/`,
`setup/`, and -- critically -- `sandbox/` itself, so pointing `source_directory` at the repo root
directly isn't safe (QAChat would try to mirror `sandbox/` into itself) or useful (it would sweep in
this benchmark's own harness as noise). Symlinking the two together doesn't work either -- QAChat's own
repository scanner explicitly refuses to follow symlinked directories, as a safety guard.

So the cell below copies just `qachat/` and `tutorials/` into a combined staging folder under
`benchmarks/sandbox/_staging/` first, and that's what gets handed to `prepare_project_workspace` as the
real source -- mirrored into a managed workspace tracked with a SHA-256 manifest, same as before, just
with one extra copy pass ahead of it.

**This is a new source path, so it gets a fresh `project_qachat_and_tutorials/` workspace** (a sibling
of the earlier package-only `project_qachat/`, now orphaned -- safe to delete once this is the working
notebook) rather than reusing the existing tuned one, the same way a fresh DeepTrack2 checkout got a
fresh workspace in notebook 90. A brand-new workspace means brand-new *default* profile settings, which
would silently drop this session's chat/embedding model tuning back to QAChat's stock defaults -- exactly
the stale-config failure mode this notebook's runs kept hitting earlier. The cell below enforces the
known-good settings explicitly right after opening the project, rather than assuming whatever
`profile.ini` happens to exist is already correct.

In [5]:
import shutil

from qachat.project import configure_project, prepare_project_workspace

staging_root = root / "benchmarks" / "sandbox" / "_staging"
combined_source = staging_root / "qachat_and_tutorials"
if combined_source.exists():
    shutil.rmtree(combined_source)
combined_source.mkdir(parents=True)

_ignore = shutil.ignore_patterns("__pycache__", "*.pyc", ".DS_Store")
shutil.copytree(root / "qachat", combined_source / "qachat", ignore=_ignore)
shutil.copytree(root / "tutorials", combined_source / "tutorials", ignore=_ignore)

workspace_root = root / "benchmarks" / "sandbox"
project = prepare_project_workspace(
    source_directory=combined_source,
    kind="code",
    workspace_root=workspace_root,
)

# A new source path can mean a brand-new workspace with default model settings --
# enforce this benchmark's known-good configuration explicitly, regardless of
# whether the workspace above was just created or already existed.
DESIRED_MODEL_SETTINGS = {
    "chat_mlx": "Qwen/Qwen3-8B-MLX-4bit",
    "chat_torch": "Qwen/Qwen3-8B",
    "embedding_mlx": "mlx-community/Qwen3-Embedding-0.6B-4bit-DWQ",
    "embedding_torch": "Qwen/Qwen3-Embedding-0.6B",
}
if any(
    getattr(project.profile, key) != value
    for key, value in DESIRED_MODEL_SETTINGS.items()
):
    project = configure_project(project.root, DESIRED_MODEL_SETTINGS)
    print("Model settings corrected to this benchmark's known-good configuration.")

print(f"Combined source staged at: {combined_source.relative_to(root)}")
print(f"Project root: {project.root.relative_to(root)}")
print(f"Project source: {project.source}")
print(f"Chat model (mlx): {project.profile.chat_mlx}")
print(f"Embedding model (mlx): {project.profile.embedding_mlx}")
print(f"Chunking: target={project.profile.target_tokens} max={project.profile.max_tokens} overlap={project.profile.overlap_tokens}")

Project root: benchmarks/sandbox/project_qachat
Project source: /Users/cmanzo/Documents/GitHub/QAChat/benchmarks/sandbox/project_qachat/qachat
Chunking: target=300 max=400 overlap=40


## 4. Open a real RAGSession

This single call is doing everything sections 5-7 in the earlier notebooks did by hand: loading the
embedding model, loading the chat model, and building (or reusing) the semantic, lexical, and code-graph
indexes. `progress_callback` prints QAChat's own structured progress events so a first run (with no
cached index yet) doesn't sit silently.


In [6]:
from qachat.rag import RAGSession

def _print_progress(event):
    if event.stage in {"ready"} or event.current is None:
        print(f"[{event.percent:5.1f}%] {event.message}")

rag_session = RAGSession(project.root, progress_callback=_print_progress)

print(f"Backend: {rag_session.backend}")
print(f"Index: {rag_session.index_status}")
print(f"Chunks: {len(rag_session.chunks)}")


[  0.0%] Opening QAChat project...
[  5.0%] Loading embedding model: mlx-community/Qwen3-Embedding-0.6B-4bit-DWQ


Fetching 13 files: 100%|██████████| 13/13 [00:00<00:00, 2660.06it/s]


[ 15.0%] Scanning selected project files...
[ 28.0%] Validating saved index metadata and source manifest...
[ 32.0%] Source unchanged; loading the saved retrieval index...
[ 85.0%] Saved retrieval index loaded.
[ 90.0%] Loading chat model: Qwen/Qwen3-8B-MLX-4bit


Fetching 7 files: 100%|██████████| 7/7 [00:00<00:00, 3045.02it/s]


[ 97.0%] Loading saved conversation...
[100.0%] QAChat ready. Index reused.
Backend: mlx
Index: reused
Chunks: 353


## 5. Try a stronger abstention instruction (notebook-only override)

The abstention questions failed in every earlier run of this notebook (0/6, either confident
fabrication or an evasive non-answer) using the default `_SYSTEM_PROMPT`. A since-removed control run
(the same six questions asked with retrieval skipped entirely) showed a more explicit, fixed-token
instruction fixes that failure in the zero-evidence case on its own (6/6, up from 0/6) -- evidence this
is a prompt-following gap, not a raw capability gap that would need a bigger chat model. A follow-up
real run confirmed the prompt fix alone doesn't carry over once retrieval hands the model real,
retrieved-but-possibly-irrelevant evidence (still 0/6 on the 0.6B model). Swapping to the larger
`Qwen/Qwen3-4B` chat model (`profile.ini`) with this override still active got real abstention to 4/6 --
so this cell is now used to separate the two variables: `USE_STRONGER_PROMPT` below toggles the override
off so this run tests the bigger model alone, against QAChat's original shipped prompt, isolating
whether the 4/6 came from the model or from the prompt override riding along with it.

This overrides `rag_session.chat_session._system_prompt` for the rest of this run only, when enabled.
`ChatSession` reads `self._system_prompt` fresh on every `reply()` call (`qachat/chat.py:490,496`), so
this takes effect for every question asked from here on without editing any `qachat/` file --
`qachat/rag.py`'s `_SYSTEM_PROMPT` constant is untouched, and a fresh `RAGSession` in a new notebook run
would still load the original regardless of this toggle. The active prompt is also recorded in the saved results (section 11) so a run using this
override is distinguishable from one that isn't.


In [7]:
USE_STRONGER_PROMPT = True  # toggle: True = override with the instruction below, False = QAChat's original _SYSTEM_PROMPT

_STRONGER_SYSTEM_PROMPT = (
    "Answer only from the supplied project evidence. Put source numbers such "
    "as [1] and [2] directly after supported claims. Do not write a separate "
    "reference list or copy bibliographic text. If the evidence clearly does not "
    "address the question, respond with exactly: Insufficient evidence. "
    "Do not guess or answer from general knowledge."
)

if USE_STRONGER_PROMPT:
    # Overrides the chat session's stored prompt directly -- ChatSession re-reads
    # self._system_prompt on every reply() call, so this is a notebook-only,
    # run-scoped change. qachat/rag.py's _SYSTEM_PROMPT constant is untouched.
    rag_session.chat_session._system_prompt = _STRONGER_SYSTEM_PROMPT
    print("Chat session system prompt overridden for this run:")
    print(_STRONGER_SYSTEM_PROMPT)
else:
    print("USE_STRONGER_PROMPT is False -- using QAChat's original _SYSTEM_PROMPT for this run:")
    print(rag_session.chat_session._system_prompt)


Chat session system prompt overridden for this run:
Answer only from the supplied project evidence. Put source numbers such as [1] and [2] directly after supported claims. Do not write a separate reference list or copy bibliographic text. If the evidence clearly does not address the question, respond with exactly: Insufficient evidence. Do not guess or answer from general knowledge.


## 6. Check gold citations against the real, chunked corpus

Before asking anything, confirm each question's gold citation(s) actually land inside a real chunk --
if a citation points at a line range no chunk covers (a stale line number after a refactor, say), that
question can't be scored fairly and is worth catching now rather than after every question has already
run.


In [8]:
from benchmarks.common import gold_evidence_indices

class _ChunkAsEvidence:
    """Adapt a real Chunk to the same (path, start_line, end_line, cell,
    citation) shape gold_evidence_indices expects from an Evidence --
    citation is unused for this check, only path/cell/line overlap matters.
    cell matters for a notebook-derived chunk: every cell's line numbers
    start at 1, so a gold citation naming a specific cell ("path#cell:1-8")
    needs the real cell number to disambiguate, not just the line range.
    """

    __slots__ = ("citation", "path", "start_line", "end_line", "cell")

    def __init__(self, chunk):
        self.citation = 0
        self.path = chunk.path
        self.start_line = chunk.start_line
        self.end_line = chunk.end_line
        self.cell = chunk.cell


chunk_evidence = [_ChunkAsEvidence(chunk) for chunk in rag_session.chunks]

unmatched = []
for task in tasks:
    if task.requires_abstention:
        continue
    if not gold_evidence_indices(chunk_evidence, task.gold_citations):
        unmatched.append(task.task_id)

if unmatched:
    print(f"WARNING: {len(unmatched)} question(s) have a gold citation that matches no real chunk: {unmatched}")
    print("Check these against the current qachat/ source before trusting their scores below.")
else:
    print(f"All {len(tasks) - sum(1 for t in tasks if t.requires_abstention)} answerable questions' gold citations match a real chunk.")

All 10 answerable questions' gold citations match a real chunk.


## 7. Ask every question for real

`rag_session.reset()` before each question keeps every turn an independent, single-turn conversation --
otherwise QAChat's normal multi-turn memory would carry context from one question into the next, which
would make later answers easier for reasons that have nothing to do with retrieval or generation
quality on that question alone. `rag_session.turns[-1].evidence` after `ask()` is the exact, real
`Evidence` QAChat retrieved and supplied for that answer -- not re-derived or approximated, the actual
objects the citation numbers in the answer refer to.


In [9]:
from tqdm.auto import tqdm

run_results: list[dict[str, object]] = []

for task in tqdm(tasks, desc="Asking", unit="question"):
    rag_session.reset()
    answer = rag_session.ask(task.question)
    turn = rag_session.turns[-1]
    run_results.append(
        {
            "task_id": task.task_id,
            "question": task.question,
            "answer": answer,
            "evidence": [
                {
                    "citation": item.citation,
                    "path": str(item.path),
                    "start_line": item.start_line,
                    "end_line": item.end_line,
                    "symbol": item.symbol,
                    "text": item.text,
                    "cell": item.cell,
                }
                for item in turn.evidence
            ],
        }
    )

print(f"Asked {len(run_results)} questions.")

Asking: 100%|██████████| 16/16 [01:05<00:00,  4.11s/question]

Asked 16 questions.


## 8. Score Tier 1 (mechanical)

Citation validity and coverage need the real `Evidence` objects each answer was actually given, so
these run against `run_results` from section 7, not against the gold set alone. A3 is excluded from
the verbatim-copy check -- it's the one question in this set where quoting the source (the system
prompt itself) verbatim is the correct answer, not a violation.


In [10]:
from dataclasses import dataclass as _dataclass
from pathlib import Path as _Path

from benchmarks.common import (
    abstains,
    citation_coverage,
    citation_validity,
    verbatim_overlap_chars,
)

VERBATIM_CHAR_THRESHOLD = 60  # a shorter shared run is plausibly just a common phrase/identifier


@_dataclass
class _EvidenceView:
    citation: int
    path: _Path
    start_line: int
    end_line: int
    symbol: object
    text: str
    cell: int | None = None


tasks_by_id = {task.task_id: task for task in tasks}
tier1_rows: list[dict[str, object]] = []

for row in run_results:
    task = tasks_by_id[row["task_id"]]
    evidence = [_EvidenceView(**item) for item in row["evidence"]]

    valid, invalid_indices = citation_validity(row["answer"], evidence_count=len(evidence))
    coverage = (
        None
        if task.requires_abstention
        else citation_coverage(row["answer"], evidence, task.gold_citations)
    )
    abstained = abstains(row["answer"])
    verbatim_chars = (
        None
        if task.task_id == "A3"
        else verbatim_overlap_chars(row["answer"], evidence)
    )

    tier1_rows.append(
        {
            "task_id": task.task_id,
            "requires_abstention": task.requires_abstention,
            "citation_valid": valid,
            "invalid_citation_indices": invalid_indices,
            "citation_coverage": coverage,
            "abstained": abstained,
            "abstention_correct": (
                (abstained if task.requires_abstention else not abstained)
            ),
            "verbatim_overlap_chars": verbatim_chars,
            "verbatim_flag": (
                verbatim_chars is not None and verbatim_chars >= VERBATIM_CHAR_THRESHOLD
            ),
        }
    )

valid_count = sum(1 for r in tier1_rows if r["citation_valid"])
abstention_correct_count = sum(1 for r in tier1_rows if r["abstention_correct"])
coverage_values = [r["citation_coverage"] for r in tier1_rows if r["citation_coverage"] is not None]
verbatim_flags = [r["task_id"] for r in tier1_rows if r["verbatim_flag"]]

print(f"Citation validity: {valid_count}/{len(tier1_rows)} answers cite only real evidence indices.")
print(f"Abstention correctness: {abstention_correct_count}/{len(tier1_rows)} (2 of these are the deliberate negatives).")
if coverage_values:
    print(f"Mean citation coverage (answerable questions with retrieved gold evidence, n={len(coverage_values)}): {sum(coverage_values) / len(coverage_values):.2f}")
else:
    print("No answerable question had its gold evidence retrieved -- coverage undefined.")
print(f"Verbatim-copy flags (>= {VERBATIM_CHAR_THRESHOLD} chars, excluding A3): {verbatim_flags or 'none'}")

Citation validity: 16/16 answers cite only real evidence indices.
Abstention correctness: 14/16 (2 of these are the deliberate negatives).
Mean citation coverage (answerable questions with retrieved gold evidence, n=8): 0.85
Verbatim-copy flags (>= 60 chars, excluding A3): none


## 9. Export the Tier 2 grading worksheet

Faithfulness (does each cited claim actually hold up against the evidence it cites) and correctness
(does the answer satisfy its fact checklist) need a human reading the answer, not a regex -- no
Claude-API judge call is wired in here, by design (see the chat discussion: an API judge is a real,
ongoing cost for a benchmark meant to be re-run often, and Tier 1 already catches the cheap mechanical
failures). This just lays out everything needed to grade Tier 2 by hand, question by question.


In [11]:
worksheet_lines = []
for row, tier1 in zip(run_results, tier1_rows):
    task = tasks_by_id[row["task_id"]]
    worksheet_lines.append(f"## {task.task_id}{' [abstention]' if task.requires_abstention else ''}")
    worksheet_lines.append(f"**Question:** {task.question}")
    worksheet_lines.append("")
    worksheet_lines.append(f"**Answer:**\n{row['answer']}")
    worksheet_lines.append("")
    worksheet_lines.append("**Evidence supplied:**")
    for item in row["evidence"]:
        symbol = f" ({item['symbol']})" if item["symbol"] else ""
        worksheet_lines.append(f"- [{item['citation']}] {item['path']}:{item['start_line']}-{item['end_line']}{symbol}")
    worksheet_lines.append("")
    worksheet_lines.append("**Fact checklist:**")
    for fact in task.fact_checklist:
        worksheet_lines.append(f"- [ ] {fact}")
    worksheet_lines.append("")
    if task.reference_answer:
        worksheet_lines.append(
            "**Reference reply (Claude's, from the fact checklist -- a comparison "
            "aid, not ground truth on its own):**"
        )
        worksheet_lines.append(task.reference_answer)
        worksheet_lines.append("")
    worksheet_lines.append(
        f"**Tier 1 (automatic):** citation_valid={tier1['citation_valid']}, "
        f"citation_coverage={tier1['citation_coverage']}, abstained={tier1['abstained']}, "
        f"abstention_correct={tier1['abstention_correct']}, "
        f"verbatim_overlap_chars={tier1['verbatim_overlap_chars']}"
    )
    worksheet_lines.append("")
    worksheet_lines.append("**Tier 2 (your read):** faithful? [ ]  checklist satisfied? [ ]  notes: ")
    worksheet_lines.append("\n---\n")

worksheet_text = "\n".join(worksheet_lines)
worksheet_path = root / "benchmarks" / "results"
worksheet_path.mkdir(parents=True, exist_ok=True)


## 10. Save results

Tier 1 scores and the raw per-question data go into `benchmarks/results/*.json`, same pattern as every
other notebook in this series. The Tier 2 worksheet is saved alongside it as its own markdown file,
timestamped the same way, so a graded copy can be diffed against the run it was graded from.


In [12]:
import json as _json
from benchmarks.common import local_run_timestamp

results_dir = root / "benchmarks" / "results"
results_dir.mkdir(parents=True, exist_ok=True)

timestamp, stamp = local_run_timestamp()

run_record = {
    "benchmark": "QAChat dogfooding -- own repo, hand-authored gold questions (Area A: code)",
    "timestamp": timestamp,
    "chat_model": project.profile.chat_model(rag_session.backend),
    "embedding_model": project.profile.embedding_model(rag_session.backend),
    "backend": rag_session.backend,
    "index_status": rag_session.index_status,
    "chunk_count": len(rag_session.chunks),
    "question_count": len(tasks),
    "abstention_question_count": sum(1 for t in tasks if t.requires_abstention),
    "gold_citation_unmatched": unmatched,
    "system_prompt": rag_session.chat_session._system_prompt,
    "tier1_summary": {
        "citation_valid_count": valid_count,
        "abstention_correct_count": abstention_correct_count,
        "mean_citation_coverage": (
            sum(coverage_values) / len(coverage_values) if coverage_values else None
        ),
        "verbatim_flagged_task_ids": verbatim_flags,
    },
    "per_question": [
        {**row, "tier1": tier1}
        for row, tier1 in zip(run_results, tier1_rows)
    ],
}

output_path = results_dir / f"{stamp}_dogfood_code.json"
output_path.write_text(_json.dumps(run_record, indent=2))

worksheet_output_path = results_dir / f"{stamp}_dogfood_code_tier2_worksheet.md"
worksheet_output_path.write_text(worksheet_text)

print(f"Saved: {output_path.relative_to(root)}")
print(f"Saved: {worksheet_output_path.relative_to(root)}")


Saved: benchmarks/results/20260929T232035+0200_dogfood_code.json
Saved: benchmarks/results/20260929T232035+0200_dogfood_code_tier2_worksheet.md


## 11. Release the models


In [13]:
rag_session.close()

print("Chat and embedding models released.")


Chat and embedding models released.
